In [6]:
import pandas as pd

df_monthly = pd.read_csv("Monthly_Rainfall_Rathnapura.csv")
df_days = pd.read_csv("Annual_Rain_Inference_Days_Rathnapura.csv")

print("=== Monthly Rainfall Columns ===")
print(df_monthly.columns.tolist())
print("\n=== Rainy Days Columns ===")
print(df_days.columns.tolist())

print("\n=== Monthly Rainfall First 3 Rows ===")
print(df_monthly.head(3))

print("\n=== Rainy Days All Rows ===")
print(df_days)

=== Monthly Rainfall Columns ===
['Date', 'Rainfall_Rathnapura']

=== Rainy Days Columns ===
['Year\t', 'No of Rainy Days_Rathnapura']

=== Monthly Rainfall First 3 Rows ===
      Date Rainfall_Rathnapura
0  1995-01                 195
1  1995-02                  84
2  1995-03                 131

=== Rainy Days All Rows ===
      Year\t  No of Rainy Days_Rathnapura
0  1961-1990                          205
1  1991-2010                          223
2  2011-2020                          225
3       2021                          198
4       2022                          226
5       2023                          244
6       2024                          236


In [1]:
pip install scikit-fuzzy

In [8]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ================== CONFIGURATION ==================
purchase_years = [2020, 2021, 2022, 2023, 2024, 2025]   # ← Change this list as you like
SIMS = 10000
SEED = 42
# =================================================

print("🔄 Loading datasets...")
df_monthly = pd.read_csv("Monthly_Rainfall_Rathnapura.csv")
df_days = pd.read_csv("Annual_Rain_Inference_Days_Rathnapura.csv")

# Clean column names (matches your screenshots)
df_monthly.columns = df_monthly.columns.str.strip().str.replace('\ufeff', '', regex=False)
df_days.columns = df_days.columns.str.strip().str.replace('\ufeff', '', regex=False)

df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format='%Y-%m', errors='coerce')
df_monthly['Rainfall_Rathnapura'] = pd.to_numeric(df_monthly['Rainfall_Rathnapura'], errors='coerce')
df_monthly = df_monthly.dropna(subset=['Date', 'Rainfall_Rathnapura']).reset_index(drop=True)
df_monthly['Year'] = df_monthly['Date'].dt.year.astype(int)

RAINY_COL = 'No of Rainy Days_Rathnapura'
df_days[RAINY_COL] = pd.to_numeric(df_days[RAINY_COL], errors='coerce')

print(f"✅ Datasets loaded! Rainfall years: {df_monthly['Year'].min()}–{df_monthly['Year'].max()}")

# ================== FUZZY SYSTEM (unchanged) ==================
rainfall_annual = ctrl.Antecedent(np.arange(0, 5001, 1), 'rainfall_annual')
rain_interf_days = ctrl.Antecedent(np.arange(0, 366, 1), 'rain_interf_days')
beta_out = ctrl.Consequent(np.arange(1.00, 1.41, 0.01), 'beta', defuzzify_method='centroid')

rainfall_annual['low'] = fuzz.trapmf(rainfall_annual.universe, [0, 0, 1000, 1750])
rainfall_annual['medium'] = fuzz.trapmf(rainfall_annual.universe, [1000, 1750, 2500, 3000])
rainfall_annual['high'] = fuzz.trapmf(rainfall_annual.universe, [2500, 3000, 5000, 5000])

rain_interf_days['low'] = fuzz.trapmf(rain_interf_days.universe, [0, 0, 30, 60])
rain_interf_days['medium'] = fuzz.trapmf(rain_interf_days.universe, [30, 60, 90, 150])
rain_interf_days['high'] = fuzz.trapmf(rain_interf_days.universe, [90, 150, 365, 365])

beta_out['no_loading'] = fuzz.trimf(beta_out.universe, [1.00, 1.00, 1.10])
beta_out['low_loading'] = fuzz.trimf(beta_out.universe, [1.00, 1.10, 1.20])
beta_out['mod_loading'] = fuzz.trimf(beta_out.universe, [1.10, 1.20, 1.30])
beta_out['high_loading'] = fuzz.trimf(beta_out.universe, [1.20, 1.30, 1.38])
beta_out['vhigh_loading'] = fuzz.trimf(beta_out.universe, [1.30, 1.38, 1.38])

rule1 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['low'], beta_out['low_loading'])
rule2 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule3 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['high'], beta_out['mod_loading'])
rule4 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['low'], beta_out['no_loading'])
rule5 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule6 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['high'], beta_out['high_loading'])
rule7 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['low'], beta_out['low_loading'])
rule8 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['medium'], beta_out['high_loading'])
rule9 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['high'], beta_out['vhigh_loading'])

beta_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
beta_sim = ctrl.ControlSystemSimulation(beta_ctrl)

# ================== MULTI-YEAR CALCULATION ==================
results = []
rng = np.random.default_rng(SEED)

for purchase_year in purchase_years:
    end_year = purchase_year - 1
    print(f"\n📅 Calculating for purchase year {purchase_year} (using data ≤ {end_year})...")
    
    # Filter rainfall
    df_hist = df_monthly[df_monthly['Year'] <= end_year].copy()
    annual_rain = df_hist.groupby('Year')['Rainfall_Rathnapura'].sum().reset_index()
    
    # Build rainy days dictionary (exactly as in your thesis)
    rainy_days_dict = {}
    rainy_days_dict.update({y: 205 for y in range(1961, 1991)})   # 1961-1990
    rainy_days_dict.update({y: 223 for y in range(1991, 2011)})   # 1991-2010
    rainy_days_dict.update({y: 225 for y in range(2011, 2021)})   # 2011-2020
    
    for _, row in df_days.iterrows():
        year_str = str(row['Year']).strip()
        if '-' not in year_str:
            try:
                y = int(float(year_str))
                if y <= end_year:
                    rainy_days_dict[y] = int(row[RAINY_COL])
            except:
                pass
    
    # Prepare lists for simulation
    years = annual_rain['Year'].tolist()
    rainfalls = annual_rain['Rainfall_Rathnapura'].tolist()
    rainy_days_list = [rainy_days_dict.get(y, 223) for y in years]
    
    # Bootstrap Monte Carlo for Expected β
    beta_samples = []
    for _ in range(SIMS):
        idx = rng.integers(0, len(years))
        rain = float(rainfalls[idx])
        days = float(rainy_days_list[idx])
        beta_sim.input['rainfall_annual'] = rain
        beta_sim.input['rain_interf_days'] = days
        beta_sim.compute()
        beta_samples.append(beta_sim.output['beta'])
    
    expected_beta = np.mean(beta_samples)
    results.append({'Purchase Year': purchase_year,
                    'Data used up to': end_year,
                    'Expected β': round(expected_beta, 4),
                    'Simulations': SIMS})

# ================== FINAL RESULTS TABLE ==================
df_results = pd.DataFrame(results)
print("\n" + "="*70)
print("EXPECTED WEATHER LOADING FACTOR β (MULTI-YEAR)")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)
print("✅ Done! Copy this table directly into your validation section.")

🔄 Loading datasets...
✅ Datasets loaded! Rainfall years: 1995–2025

📅 Calculating for purchase year 2020 (using data ≤ 2019)...

📅 Calculating for purchase year 2021 (using data ≤ 2020)...

📅 Calculating for purchase year 2022 (using data ≤ 2021)...

📅 Calculating for purchase year 2023 (using data ≤ 2022)...

📅 Calculating for purchase year 2024 (using data ≤ 2023)...

📅 Calculating for purchase year 2025 (using data ≤ 2024)...

EXPECTED WEATHER LOADING FACTOR β (MULTI-YEAR)
 Purchase Year  Data used up to  Expected β  Simulations
          2020             2019      1.3344        10000
          2021             2020      1.3351        10000
          2022             2021      1.3355        10000
          2023             2022      1.3362        10000
          2024             2023      1.3369        10000
          2025             2024      1.3371        10000
✅ Done! Copy this table directly into your validation section.
